# Usage | 1. Seed points
This notebook explains the seed points concept behind growbikenet with illustrative examples.  
Parameters covered: `seed_point_type`, `seed_point_linking`, `street_network_file`, `seed_point_file`, `seed_point_tags`

## Network construction through linking seed points

Growbikenet is based on the idea of linking seed points, inspired by Dutch CROW Design manual for bicycle traffic. These are points in a city that shall be connected by protected bicycle infrastructure, yielding edges in a bicycle network. The choice and placement of seed points is crucial. To build a well-covering bicycle network, seed points should cover most of the city. By default, in growbikenet seed points are arbitrary, city-spanning points on a grid, snapped to the street network, but it is also possible to assign actual points of interests such as railway stations, schools, or parks, see the end of this notebook. The `seed_point_type` parameter is controlling the type of seed points chosen.

![Seed points and triangulation](_static/seedpoints.jpg)

Once seed points are defined, they need to be linked. The type of linking is controlled by the `seed_point_linking` parameter. By default, growbikenet triangulates or quadrangulates the seed points automatically, building triangle or square-shaped links between the seed points. Once the linking process is done (triangulation or quadrangulation), an abstract, unrouted network between seed points is established, see the figure above. Growbikenet then builds a bicycle network on this network's edges, routed on the street network. Run the cells below for live examples of this process.

## Import growbikenet

The standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

## Save street network locally

Here we work with Barcelona. When running growbikenet, it uses OSMnx to fetch street network data from [OSM](https://www.openstreetmap.org/). OSMnx has data caching implemented, so when running it multiple times on the same city, it will load the cached json data. Unfortunately, loading json is slow. We therefore first download and save the (undirected) street network once locally, under `Barcelona_streets.gpkg`, so we can load that file fast into growbikenet whenever we will use it (because gpkg files are loaded fast):

In [ ]:
import osmnx as ox
g = ox.graph_from_place("Barcelona", network_type='all_public')
ox.io.save_graph_geopackage(g.to_undirected(), "Barcelona_streets.gpkg")

## Triangulation versus quadrangulation

This section explores the utility of the `seed_point_type` and `seed_point_linking` parameters. By default, growbikenet detects automatically the optimal type and spacing of the seed point grid, and of the seed point linking process for a given city: 

1. Many European cities like Berlin or London have very irregular, organic street networks. On these networks, growbikenet constructs by default a triangular grid (`seed_point_type=='grid_triangle'`) that it naturally triangulates (`seed_point_linking=='triangulate_delaunay'`).
2. Some cities like Prague or Budapest have some square grid elements. On these networks, growbikenet constructs by default a square grid (`seed_point_type=='grid_square'`) that it triangulates (`seed_point_linking=='triangulate_delaunay'`).
3. Many US cities, and some European cities, like Manhattan or Barcelona, have ample grid elements in their street networks. On these networks, growbikenet constructs by default a square grid (`seed_point_type=='grid_square'`) that it quadrangulates (`seed_point_linking=='quadrangulate'`).

See below the quadrangulated square grid result for Barcelona, loaded in from the downloaded file via the `street_network_file` parameter:

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                               street_network_file="Barcelona_streets.gpkg",)

Explore the results interactively:

In [ ]:
edges_ranked.explore(tiles="CartoDB Positron", 
                     style_kwds={"weight": 3, "color": "#096a51"},)

Compare this to the output if we were to instruct growbikenet to instead triangulate on the square grid instead of quadrangulate. It features some ugly zig-zags:

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                               seed_point_linking ='triangulate_delaunay',
                               street_network_file="Barcelona_streets.gpkg",)

In [ ]:
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"},)

Under the hood, growbikenet also automatically adapts the spacing of the seed point grid.

## Spacing of the seed point grid

In the above network outputs, observe how the grid is more finegrained in the quadrangulation than in the triangulation. This is because by default, `seed_point_grid_spacing` is set to `auto`, leading growbikenet to automatically adjust the grid spacing in a way to ensure that any point in the city is always within 500m of the network (under perfect conditions). Concretely, the spacing of seed points is every 1707m in the triangulation case, but ony 1000m in the quadrangulation case.

Let's get back to the nice quadrangulated square grid of Barcelona. As a big part of the city has a square grid structure, each 3x3 blocks could be considered as a "superblock", with a bike network going around it. Let us now try to choose a spacing manually that will create this network:

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_grid_spacing=530,
                              street_network_file="Barcelona_streets.gpkg",)

A spacing of 530m is much tighter, but fits the "superblocks" much nicer - zoom into the map to see the details:

In [ ]:
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"},)

## Preset seed points: Rail, school, park

Apart from connecting street intersection seed points from an arbitrary grid, growbikenet also supports connecting seed points that are actual points of interests like rail stations, schools, or parks:

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_type="rail",
                              street_network_file="Barcelona_streets.gpkg",)
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"})

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_type="school",
                              street_network_file="Barcelona_streets.gpkg",)
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"})

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_type="park",
                              street_network_file="Barcelona_streets.gpkg",)
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"})

## Custom seed points: file, tags

Seed points can be completely custom, for example loaded from file. Here, we first save the coordinates of a bunch of touristic spots in Barcelona as gpkg file, to then load it to construct a bicycle network around them:

In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.DataFrame(
    {
        "Latitude": [41.403653, 41.378289, 41.361902, 41.413509, 41.409700, 41.399190, 41.436646, 41.391843, 41.395832],
        "Longitude": [2.173703, 2.192716, 2.158304, 2.153073, 2.126480, 2.212746, 2.191520, 2.165109, 2.112801],
    }
)
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude), crs="EPSG:4326"
)
gdf.to_file("Barcelona_touristic_spots.gpkg")

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_type="file",
                              seed_point_file="Barcelona_touristic_spots.gpkg",
                              street_network_file="Barcelona_streets.gpkg",)
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"})

Finally, we are able to use arbitrary OSM tags. For example, to create a bike network between all ice cream shops and all places of worship:

In [ ]:
edges_ranked = gbn.growbikenet("Barcelona",
                              seed_point_type="tags",
                              seed_point_tags={"amenity": ["ice_cream", "place_of_worship"]},
                              street_network_file="Barcelona_streets.gpkg",)
edges_ranked.explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 3, "color": "#096a51"})

Note that under the hood, the seed point types `rail`, `school`, and `park` are just preset tags. For example, instead of selecting railway stations via `seed_point_type="rail"`, they could be selected equivalently via `seed_point_type="tags"` and `seed_point_tags={"railway": ["station", "halt"]}`.